# landing_eda_receita_federal — EDA Receita Federal (CNPJ)

> **Exploration only — not part of the production pipeline.**

## Purpose
Understand structure, encoding, and anomalies in the Receita Federal ZIP/CSV files
before defining and validating the Bronze schema.

## Central question
*"What is the structure of the RF raw files, what are the column layouts
(no headers!), what anomalies exist, and what does Bronze need to handle?"*

## Engine
DuckDB reads ZIP/CSV directly — same as Bronze pipeline. No pandas except for
the initial CSV read (RF files need explicit encoding handling).

## Notes
CSVs have no headers — column names assigned per official RF layout.
Socios excluded — ADR-005 (partial CPF under LGPD).

## Steps
1. Imports and configuration
2. File inventory + ZIP integrity
3. Domain tables
4. Empresas
5. Estabelecimentos
6. Simples
7. Inconsistencies and design decisions
8. Dynamic summary


## Step 1 — Imports and configuration

In [ ]:
# =============================================================================
# Step 1 — Imports and configuration
# =============================================================================

import sys
import zipfile
from pathlib import Path

# --- Resolve root ------------------------------------------------------------
try:
    _nb_file = Path(__vsc_ipynb_file__).resolve()
    _root_candidate = _nb_file.parent.parent.parent  # eda/ → notebooks/ → fornecedor360-local/
except NameError:
    _root_candidate = Path.cwd()
    for candidate in [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent, Path.cwd().parent.parent.parent]:
        if (candidate / "utils" / "paths.py").exists():
            _root_candidate = candidate
            break

sys.path.insert(0, str(_root_candidate / "utils"))

from paths        import get_project_root, to_sql_path
from duckdb_utils import get_connection

PROJECT_ROOT = get_project_root()
con = get_connection()  # in-memory — EDA never writes

RF_ROOT = PROJECT_ROOT / "data" / "raw" / "receita_federal"
months  = sorted([d for d in RF_ROOT.iterdir() if d.is_dir()]) if RF_ROOT.exists() else []

SAMPLE_MONTH = months[-1].name if months else None
SAMPLE_DIR   = RF_ROOT / SAMPLE_MONTH if SAMPLE_MONTH else None

print(f"PROJECT_ROOT  : {PROJECT_ROOT}")
print(f"RF_ROOT       : {RF_ROOT}")
print(f"Months found  : {len(months)}")
print(f"Sample month  : {SAMPLE_MONTH}")

## Step 2 — File inventory + ZIP integrity

**Question:** *Are all 27 ZIP files present for the sample month?
Are any corrupted? What is the size distribution?*


In [ ]:
zips = sorted(SAMPLE_DIR.glob("*.zip")) if SAMPLE_DIR else []

print(f"=== File Inventory + ZIP Integrity — {SAMPLE_MONTH} ===")
print(f"Total zip files: {len(zips)}")
print()
print(f"{'file':<35} {'size_mb':>8} {'status':>10}")
print("-" * 57)

corrupted = []
for zp in zips:
    size_mb = zp.stat().st_size / 1e6
    try:
        with zipfile.ZipFile(zp) as zf:
            bad = zf.testzip()
        status = "OK" if bad is None else f"BAD:{bad[:20]}"
        if bad:
            corrupted.append(zp.name)
    except zipfile.BadZipFile:
        status = "CORRUPT"
        corrupted.append(zp.name)
    print(f"{zp.name:<35} {size_mb:>8.1f} {status:>10}")

print()
if corrupted:
    print(f"[WARN] {len(corrupted)} corrupted: {corrupted}")
else:
    print("[OK] All ZIP files valid")


## Step 3 — Domain tables

**Question:** *What reference tables exist (Naturezas, CNAEs, Municípios, Países, Motivos)?
How many codes do they have? Are they stable across months?*


In [ ]:
import io, pandas as pd

def read_zip_csv(zip_path, sep=";", encoding="latin-1", nrows=None):
    """Read CSV from ZIP into pandas DataFrame — for EDA of RF files without headers."""
    with zipfile.ZipFile(zip_path) as zf:
        csv_bytes = zf.read(zf.namelist()[0])
    for enc in [encoding, "utf-8", "cp1252"]:
        try:
            return pd.read_csv(io.BytesIO(csv_bytes), sep=sep, encoding=enc,
                               header=None, nrows=nrows, dtype=str, low_memory=False)
        except UnicodeDecodeError:
            continue
    raise ValueError(f"Could not decode {zip_path.name}")

print("=== Domain Tables ===")
print()

domain_files = {
    "Naturezas"     : ("Naturezas.zip",     ["natureza_codigo", "natureza_descricao"]),
    "CNAEs"         : ("Cnaes.zip",         ["cnae_codigo", "cnae_descricao"]),
    "Municípios"    : ("Municipios.zip",     ["municipio_codigo", "municipio_descricao"]),
    "Países"        : ("Paises.zip",         ["pais_codigo", "pais_descricao"]),
    "Motivos"       : ("Motivos.zip",        ["motivo_codigo", "motivo_descricao"]),
    "Qualificações" : ("Qualificacoes.zip",  ["qualificacao_codigo", "qualificacao_descricao"]),
}

print(f"{'table':<15} {'rows':>6}  sample key")
print("-" * 45)
for name, (fname, cols) in domain_files.items():
    fpath = SAMPLE_DIR / fname
    if not fpath.exists():
        print(f"{name:<15} [MISSING]")
        continue
    df = read_zip_csv(fpath)
    df.columns = cols + [f"col{i}" for i in range(len(df.columns) - len(cols))]
    n = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]
    sample_key = con.execute(f"SELECT {cols[0]} FROM df LIMIT 1").fetchone()[0]
    print(f"{name:<15} {n:>6}  {sample_key}")


## Step 4 — Empresas

**Question:** *What is the structure of the Empresas files (no headers)?
What is the porte_empresa distribution? Is cnpj_basico always 8 digits?*


In [ ]:
EMPRESAS_COLS = [
    "cnpj_basico", "razao_social", "natureza_juridica",
    "qualificacao_responsavel", "capital_social",
    "porte_empresa", "ente_federativo_responsavel",
]

emp_path = SAMPLE_DIR / "Empresas0.zip"
df = read_zip_csv(emp_path)
df.columns = EMPRESAS_COLS[:len(df.columns)]
total_emp = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]

print(f"=== Empresas (Empresas0.zip) — {total_emp:,} rows ===")
print()

print("--- cnpj_basico (should always be 8 digits) ---")
con.execute("""
    SELECT
        COUNT(DISTINCT cnpj_basico)                               AS distinct_cnpj,
        COUNT(*) FILTER (WHERE LENGTH(cnpj_basico) != 8)          AS non_8digit,
        COUNT(*) FILTER (WHERE regexp_matches(cnpj_basico, '[A-Za-z]')) AS alphanumeric
    FROM df
""").df()

print()
print("--- porte_empresa distribution ---")
con.execute("""
    SELECT porte_empresa, COUNT(*) AS cnt
    FROM df GROUP BY porte_empresa ORDER BY cnt DESC
""").df()


## Step 5 — Estabelecimentos

**Question:** *What is the structure of Estabelecimentos? What is the situacao_cadastral
distribution? How is the full 14-digit CNPJ assembled from basico+ordem+dv?
What is the UF distribution?*


In [ ]:
ESTAB_COLS = [
    "cnpj_basico", "cnpj_ordem", "cnpj_dv",
    "identificador_matriz_filial", "nome_fantasia",
    "situacao_cadastral", "data_situacao_cadastral",
    "motivo_situacao_cadastral", "nome_cidade_exterior",
    "pais", "data_inicio_atividade",
    "cnae_fiscal_principal", "cnae_fiscal_secundaria",
    "tipo_logradouro", "logradouro", "numero",
    "complemento", "bairro", "cep", "uf", "municipio",
    "ddd_1", "telefone_1", "ddd_2", "telefone_2",
    "ddd_fax", "fax", "correio_eletronico",
    "situacao_especial", "data_situacao_especial",
]

estab_path = SAMPLE_DIR / "Estabelecimentos0.zip"
df = read_zip_csv(estab_path, nrows=100_000)
df.columns = ESTAB_COLS[:len(df.columns)]

total_estab = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]
print(f"=== Estabelecimentos (Estabelecimentos0.zip, sample 100k) — {total_estab:,} rows ===")
print()

print("--- situacao_cadastral distribution ---")
con.execute("""
    SELECT situacao_cadastral, COUNT(*) AS cnt
    FROM df GROUP BY situacao_cadastral ORDER BY cnt DESC
""").df()

print()
print("--- UF distribution (top 10) ---")
con.execute("""
    SELECT uf, COUNT(*) AS cnt FROM df GROUP BY uf ORDER BY cnt DESC LIMIT 10
""").df()

print()
print("--- CNPJ assembly check (basico || ordem || dv = 14 chars) ---")
con.execute("""
    SELECT COUNT(*) FILTER (WHERE LENGTH(cnpj_basico || cnpj_ordem || cnpj_dv) = 14) AS valid_14,
           COUNT(*) AS total FROM df
""").df()


## Step 6 — Simples

**Question:** *What is the opcao_simples and opcao_mei distribution?
What is the sentinel value 00000000 in date fields and how should Bronze handle it?*


In [ ]:
SIMPLES_COLS = [
    "cnpj_basico", "opcao_simples", "data_opcao_simples",
    "data_exclusao_simples", "opcao_mei",
    "data_opcao_mei", "data_exclusao_mei",
]

simples_path = SAMPLE_DIR / "Simples.zip"
df = read_zip_csv(simples_path)
df.columns = SIMPLES_COLS[:len(df.columns)]
total_sim = con.execute("SELECT COUNT(*) FROM df").fetchone()[0]

print(f"=== Simples — {total_sim:,} rows ===")
print()

print("--- opcao_simples distribution ---")
con.execute("SELECT opcao_simples, COUNT(*) AS cnt FROM df GROUP BY opcao_simples ORDER BY cnt DESC").df()

print()
print("--- Sentinel value 00000000 in date fields ---")
for col in ["data_opcao_simples", "data_exclusao_simples", "data_opcao_mei", "data_exclusao_mei"]:
    result = con.execute(f"""
        SELECT
            COUNT(*) FILTER (WHERE {col} = '00000000') AS sentinel_00,
            COUNT(*) FILTER (WHERE {col} IS NULL OR {col} = '') AS nulls,
            COUNT(*) AS total FROM df
    """).fetchone()
    pct = result[0] / result[2] * 100 if result[2] else 0
    print(f"  {col:<30} sentinel=00: {result[0]:>8,} ({pct:.1f}%)  null: {result[1]:>8,}")

print()
print("[NOTE] Bronze: preserve 00000000 as-is. Silver: TRY_STRPTIME returns NULL for sentinel.")


In [ ]:
print("--- Tipos inferidos pelo DuckDB ao ler Simples.zip ---")
print("[CRITICAL] Estes são os tipos que Bronze vai gravar no Parquet")
print()

# Lê o arquivo CSV interno do ZIP via zipfile + DuckDB in-memory
import io, pandas as pd

simples_zip = SAMPLE_DIR / "Simples.zip"
with zipfile.ZipFile(simples_zip) as zf:
    csv_name = zf.namelist()[0]
    csv_bytes = zf.read(csv_name)

# DuckDB lê o CSV com schema auto-inferido — mesmo comportamento do Bronze
# sem forçar dtype=str como o pandas faz
tmp_path = to_sql_path(SAMPLE_DIR / "_tmp_simples_sample.csv")
(SAMPLE_DIR / "_tmp_simples_sample.csv").write_bytes(csv_bytes[:500_000])  # primeiros 500KB bastam para inferência

inferred = con.execute(f"""
    DESCRIBE SELECT * FROM read_csv(
        '{tmp_path}',
        header=false,
        sep=';',
        columns={{
            'cnpj_basico':          'VARCHAR',
            'opcao_simples':        'VARCHAR',
            'data_opcao_simples':   'VARCHAR',
            'data_exclusao_simples':'VARCHAR',
            'opcao_mei':            'VARCHAR',
            'data_opcao_mei':       'VARCHAR',
            'data_exclusao_mei':    'VARCHAR'
        }}
    ) LIMIT 0
""").fetchall()

# Remove arquivo temporário
(SAMPLE_DIR / "_tmp_simples_sample.csv").unlink()

for row in inferred:
    flag = " <-- WARN: BIGINT date — Silver needs CASE WHEN val=0 pattern" \
           if row[1] == 'BIGINT' and 'data' in row[0] else ""
    print(f"  {row[0]:<30} {row[1]}{flag}")

print()
print("NOTE: Se Bronze gravar sem schema forçado, os tipos acima podem diferir.")
print("      Sempre confirmar com DESCRIBE após execução do Bronze notebook.")

## Step 7 — Inconsistencies and design decisions

**Question:** *What anomalies require explicit handling in Bronze?
What are the key decisions made?*


In [ ]:
print("=== Inconsistencies and Design Decisions ===")
print()

print("1. No column headers in CSV files")
print("   Column names assigned via explicit schema per official RF layout")
print("   Empresas: 7 columns | Estabelecimentos: 30 columns | Simples: 7 columns")
print()

print("2. CNPJ structure across files")
print("   Empresas/Simples: cnpj_basico (8 digits) — legal entity identifier")
print("   Estabelecimentos: cnpj_basico + cnpj_ordem + cnpj_dv — full 14-digit CNPJ")
print("   Bronze: join on cnpj_basico. Silver: concatenate to cnpj_normalized")
print()

print("3. Sentinel 00000000 in date fields (Simples)")
print("   Represents 'no date' — not a null, a sentinel value from the source system")
print("   Bronze: keep as-is (VARCHAR). Silver: TRY_STRPTIME returns NULL")
print()

print("4. capital_social — Brazilian decimal format (comma as decimal separator)")
print("   Bronze: keep as STRING. Silver: REPLACE(',', '.') before CAST to DECIMAL")
print()

print("5. Shard structure — 10 files per table")
print("   Empresas0.zip to Empresas9.zip — same schema, union via wildcard")
print("   Bronze: read_csv('Empresas*.zip') — DuckDB unions automatically")
print()

print("6. Socios excluded (ADR-005 — Privacy by Design)")
print("   Socios0-9.zip contain partial CPF — personal data under LGPD")
print("   These files are not downloaded by bootstrap_receita_federal")


## Step 8 — Dynamic summary

**Question:** *What have I learned? What does Bronze depend on?*


In [ ]:
from IPython.display import display, Markdown

n_zips = len(zips) if SAMPLE_DIR else 0
total_zip_mb = sum(zp.stat().st_size for zp in zips) / (1024*1024) if zips else 0

summary = f"""
## Receita Federal — EDA Summary

| | |
|---|---|
| Sample month | {SAMPLE_MONTH} |
| ZIP files | {n_zips} files ({total_zip_mb:.0f} MB) |
| Corrupted ZIPs | {len(corrupted) if 'corrupted' in dir() else 'N/A'} |
| Domain tables | 6 (Naturezas, CNAEs, Municípios, Países, Motivos, Qualificações) |

### Key Bronze decisions
| Decision | Reason |
|---|---|
| No headers in CSV | Explicit schema assignment required |
| All columns as STRING | No type guarantees; dates as VARCHAR |
| Sentinel 00000000 kept | Preserve raw data — Silver converts via TRY_STRPTIME |
| capital_social as STRING | Brazilian decimal format (comma separator) |
| Wildcard `Empresas*.zip` | 10 shards per table — DuckDB unions automatically |
| Socios not downloaded | ADR-005 — partial CPF is personal data under LGPD |
"""

display(Markdown(summary))
